# Transformer Language Model

Clean implementation of a character-level transformer trained on Tiny Shakespeare. Contains only the essentials:

1. **Setup & Data Loading** — read `input.txt`, build vocabulary
2. **Tokenizer** — `encode` / `decode` between strings and integer tensors
3. **Dataset Preparation** — 90/10 train/val split, `get_batch` sampler
4. **Transformer Components**:
   - `FeedForward` — per-token non-linear processing (4x expansion)
   - `SelfAttentionHead` — causal self-attention with scaling
   - `MultiHeadAttention` — parallel heads + output projection
   - `TransformerBlock` — attention + FFN with pre-norm residuals
   - `TransformerLanguageModel` — token + position embeddings → N blocks → logits
5. **Training loop** — AdamW, cross-entropy
6. **Generation** — sliding-window softmax sampling

## 1. Setup & Data Loading

In [ ]:
import torch
from torch import nn

## Device setup — use Apple Silicon GPU if available
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {device}")

## Read the input data
with open('../input.txt', 'r') as file:
    text = file.read()

print(f"Dataset size: {len(text):,} characters")

## Extract unique characters as our vocabulary
vocab = sorted(set(text))
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")

## 2. Tokenizer

Character-level: map each character to an integer and back.

In [ ]:
## Character-to-integer mappings
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for i, ch in enumerate(vocab)}

def encode(string: str) -> list[int]:
    return [stoi[c] for c in string]

def decode(ints: list[int]) -> str:
    return ''.join([itos[i] for i in ints])

## Quick test
print(encode('Hi there!'))
print(decode(encode('Hi there!')))

## 3. Dataset Preparation

Encode the full text into a tensor and split 90/10.

In [ ]:
dataset_encoding = torch.tensor(encode(text), dtype=torch.long)
print(f"Full dataset: {dataset_encoding.shape}")

train_length = round(len(text) * 0.9)
training_data = dataset_encoding[:train_length]
validation_data = dataset_encoding[train_length:]
print(f"Training:   {training_data.shape}")
print(f"Validation: {validation_data.shape}")

## 4. Hyperparameters & Batch Sampler

In [ ]:
## Hyperparameters
embed_dim  = 128   # model dimension (was 64 → bumped to 128 beats MLP)
num_heads  = 4     # head_size = embed_dim // num_heads = 32
block_size = 8     # context window (chars)
batch_size = 32
n_layers   = 4     # stacked transformer blocks

def get_batch(split: str):
    """Sample a batch. Returns x: (B, T) input, y: (B, T) targets (shifted by 1)."""
    data = training_data if split == 'train' else validation_data
    i = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[idx : idx + block_size] for idx in i])
    y = torch.stack([data[idx + 1 : idx + 1 + block_size] for idx in i])
    return x.to(device), y.to(device)

## 5. Transformer Components

Built bottom-up: `FeedForward` → `SelfAttentionHead` → `MultiHeadAttention` → `TransformerBlock` → `TransformerLanguageModel`.

### FeedForward

Per-token non-linear processing. Expands to 4x, applies ReLU, projects back. Gives the non-linearity room to compute.

In [ ]:
class FeedForward(nn.Module):
    """Adds non-linearity: Linear → ReLU → Linear with 4x expansion."""
    def __init__(self, embed_dim):
        super().__init__()
        self.ffn_linear = nn.Linear(embed_dim, 4 * embed_dim)
        self.ffn_relu   = nn.ReLU()
        self.ffn_out    = nn.Linear(4 * embed_dim, embed_dim)

    def forward(self, x):
        x_linear = self.ffn_linear(x)
        x_relu   = self.ffn_relu(x_linear)
        return self.ffn_out(x_relu)

### SelfAttentionHead

Single head of causal self-attention.
- **Q** = "what am I looking for?"
- **K** = "what do I contain?"
- **V** = "what information do I send?"

Scores are scaled by `sqrt(head_size)` to keep softmax in its sensitive region.

In [ ]:
class SelfAttentionHead(nn.Module):
    """Single head of causal self-attention."""
    def __init__(self, embed_dim, head_size):
        super().__init__()
        self.W_q = nn.Linear(embed_dim, head_size, bias=False)
        self.W_k = nn.Linear(embed_dim, head_size, bias=False)
        self.W_v = nn.Linear(embed_dim, head_size, bias=False)
        ## Pre-compute the causal mask (not a learnable parameter)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape                                          # (B, T, embed_dim)
        Q = self.W_q(x)                                            # (B, T, head_size)
        K = self.W_k(x)                                            # (B, T, head_size)
        V = self.W_v(x)                                            # (B, T, head_size)

        scores  = Q @ K.transpose(-2, -1) / (K.shape[-1] ** 0.5)   # (B, T, T)
        scores  = scores.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        weights = torch.softmax(scores, dim=-1)                    # (B, T, T)
        out     = weights @ V                                      # (B, T, head_size)
        return out, weights

### MultiHeadAttention

Run `num_heads` heads in parallel, each with `head_size = embed_dim // num_heads`. Concatenate outputs along the last dim → back to `embed_dim`, then project.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Parallel heads + output projection. head_size * num_heads = embed_dim."""
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.heads_list = nn.ModuleList([
            SelfAttentionHead(embed_dim, embed_dim // num_heads)
            for _ in range(num_heads)
        ])
        self.out_projection = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        head_out    = [head(x)[0] for head in self.heads_list]
        concat_out  = torch.cat(head_out, dim=-1)
        return self.out_projection(concat_out)

### TransformerBlock

Pre-norm style: normalize → sublayer → add residual. The residual (`x + f(x)`) gives gradients a highway through deep stacks — derivative is `1 + f'(x)`, the `+1` lets gradients flow unchanged.

In [ ]:
class TransformerBlock(nn.Module):
    """Multi-head attention + FFN, each wrapped in pre-norm + residual."""
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.heads = MultiHeadAttention(embed_dim, num_heads)
        self.ffn   = FeedForward(embed_dim)
        self.ln1   = nn.LayerNorm(embed_dim)
        self.ln2   = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.heads(self.ln1(x))   # residual around attention
        x = x + self.ffn(self.ln2(x))     # residual around FFN
        return x

### TransformerLanguageModel

```
token IDs (B, T)
  → token embedding + position embedding  (B, T, embed_dim)
  → N × TransformerBlock                    (B, T, embed_dim)
  → final LayerNorm → Linear                (B, T, vocab_size)
  → cross-entropy loss
```

In [ ]:
class TransformerLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, block_size, n_layers):
        super().__init__()
        self.embedding            = nn.Embedding(vocab_size, embed_dim)
        self.positioned_embedding = nn.Embedding(block_size, embed_dim)
        self.stacked_blocks       = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads) for _ in range(n_layers)
        ])
        self.ln  = nn.LayerNorm(embed_dim)
        self.out = nn.Linear(embed_dim, vocab_size)

    def forward(self, x, targets=None):
        x_embedded     = self.embedding(x)
        x_positioned   = x_embedded + self.positioned_embedding(
            torch.arange(x.shape[1], device=x.device)
        )
        x_transformed  = self.stacked_blocks(x_positioned)
        logits         = self.out(self.ln(x_transformed))

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = nn.functional.cross_entropy(
                logits.view(B*T, C), targets.view(B*T)
            )
        return logits, loss

## 6. Training

In [ ]:
model = TransformerLanguageModel(vocab_size, embed_dim, num_heads, block_size, n_layers).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for i in range(10000):
    x, y = get_batch('train')
    logits, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if i % 1000 == 0:
        losses = []
        for _ in range(100):
            xv, yv = get_batch('val')
            _, lv = model(xv, yv)
            losses.append(lv.item())
        print(f"step {i}: val loss {sum(losses)/len(losses):.4f}")

## 7. Generation

Sliding-window: feed the last `block_size` tokens, sample from softmax of the final position, append, repeat.

In [ ]:
def generate(model, seed_text: str, length: int) -> str:
    context = torch.tensor([encode(seed_text[-block_size:])], device=device)
    output  = []
    for _ in range(length):
        logits, _ = model(context)
        logits    = logits[:, -1, :]                       # (B, vocab_size)
        probs     = nn.functional.softmax(logits, dim=-1)
        next_idx  = torch.multinomial(probs, 1)
        output.append(next_idx.item())
        context   = torch.cat([context[:, 1:], next_idx], dim=1)
    return decode(output)

print(generate(model, "Hi King", 500))